In [1]:
pip -q install transformers accelerate sentencepiece opencv-python pillow

In [2]:
import torch
import cv2
from PIL import Image

print(torch.__version__)
print("Setup Complete ✅")

2.11.0+cu128
Setup Complete ✅


In [3]:
from transformers import AutoProcessor, AutoModelForImageTextToText
import torch

device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "Qwen/Qwen2.5-VL-3B-Instruct"

processor = AutoProcessor.from_pretrained(model_name)
model = AutoModelForImageTextToText.from_pretrained(
    model_name,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto"
)

print("✅ Model Loaded Successfully!")

preprocessor_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/1.05k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.37k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/5.70k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/216 [00:00<?, ?B/s]

✅ Model Loaded Successfully!


In [4]:
from google.colab import files

uploaded = files.upload()

Saving Screenshot_5.jpg to Screenshot_5.jpg


In [5]:
import torch
print(torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

True
Tesla T4


In [6]:
from PIL import Image

image_path = list(uploaded.keys())[0]
image = Image.open(image_path).convert("RGB")

messages = [
    {
        "role": "user",
        "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": "Describe this image in detail."}
        ],
    }
]

text = processor.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
)

inputs = processor(
    text=[text],
    images=[image],
    return_tensors="pt"
).to(model.device)

generated_ids = model.generate(
    **inputs,
    max_new_tokens=150
)

output = processor.batch_decode(
    generated_ids,
    skip_special_tokens=True
)

print(output[0])

system
You are a helpful assistant.
user
Describe this image in detail.
assistant
The image features a small kitten with striking blue eyes and a mix of gray, white, and brown fur. The kitten is standing on a light-colored surface, possibly a table or floor, against a plain, light gray background. The kitten's ears are perked up, and it appears to be looking directly at the camera with a curious and attentive expression. The overall scene is simple and clean, highlighting the kitten's adorable and innocent appearance.


In [7]:
from google.colab import files

uploaded = files.upload()

Saving Clip_05.mp4 to Clip_05.mp4


In [8]:
import cv2
import os

video_path = list(uploaded.keys())[0]

frames_dir = "video_frames"
os.makedirs(frames_dir, exist_ok=True)

cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)

frame_interval = int(fps * 2)   # one frame every 2 seconds

count = 0
saved = 0

while True:
    ret, frame = cap.read()

    if not ret:
        break

    if count % frame_interval == 0:
        frame_name = os.path.join(frames_dir, f"frame_{saved}.jpg")
        cv2.imwrite(frame_name, frame)
        saved += 1

    count += 1

cap.release()

print("Frames Extracted:", saved)

Frames Extracted: 15


In [9]:
import os
from PIL import Image

question = "What is happening in this scene?"

frame_files = sorted(os.listdir("video_frames"))

for frame in frame_files:

    image = Image.open(os.path.join("video_frames", frame)).convert("RGB")

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": question},
            ],
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = processor(
        text=[text],
        images=[image],
        return_tensors="pt"
    ).to(model.device)

    generated_ids = model.generate(
        **inputs,
        max_new_tokens=80
    )

    answer = processor.batch_decode(
        generated_ids,
        skip_special_tokens=True
    )[0]

    print("="*60)
    print(frame)
    print(answer)

frame_0.jpg
system
You are a helpful assistant.
user
What is happening in this scene?
assistant
The image shows a wooden cutting board with a few green leaves on it. The text "Spicy Tomato Garlic Rice" is displayed at the top of the image, suggesting that the leaves might be used as garnish for a dish called Spicy Tomato Garlic Rice.
frame_1.jpg
system
You are a helpful assistant.
user
What is happening in this scene?
assistant
The image shows a dish of Spicy Tomato Garlic Rice. The rice is bright orange, indicating the presence of tomatoes and possibly other spices. There is a garnish on top, which appears to be a slice of lemon or a similar citrus fruit. The dish is presented on a white plate with a pink rim. The text "Spicy Tomato Garlic Rice" is written at the top of the image
frame_10.jpg
system
You are a helpful assistant.
user
What is happening in this scene?
assistant
In the image, there is a frying pan on what appears to be a stove or cooking surface. Inside the pan, there is 

In [11]:
question = input("Ask a question about the video: ")

Ask a question about the video: What is the person doing?


In [12]:
question = input("Ask a question about the video: ")

for frame in frame_files:

    image = Image.open(os.path.join("video_frames", frame)).convert("RGB")

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": question},
            ],
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = processor(
        text=[text],
        images=[image],
        return_tensors="pt"
    ).to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=60
    )

    answer = processor.batch_decode(
        output,
        skip_special_tokens=True
    )[0]

    print("\n------------------------")
    print("Frame:", frame)
    print("Answer:", answer)

Ask a question about the video: What food is being prepared?

------------------------
Frame: frame_0.jpg
Answer: system
You are a helpful assistant.
user
What food is being prepared?
assistant
The food being prepared in the picture is Spicy Tomato Garlic Rice.

------------------------
Frame: frame_1.jpg
Answer: system
You are a helpful assistant.
user
What food is being prepared?
assistant
The food being prepared in the image is Spicy Tomato Garlic Rice.

------------------------
Frame: frame_10.jpg
Answer: system
You are a helpful assistant.
user
What food is being prepared?
assistant
The image shows a dish with red ingredients, likely tomatoes or chili peppers, being cooked in a pan. The presence of a wooden spoon and the overall setup suggests that this could be a stir-fry or a similar type of dish. The specific dish cannot be determined without more context, but it appears

------------------------
Frame: frame_11.jpg
Answer: system
You are a helpful assistant.
user
What food is 

In [13]:
question = input("Ask a question about the video: ")

for frame in frame_files:

    image = Image.open(os.path.join("video_frames", frame)).convert("RGB")

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": question},
            ],
        }
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = processor(
        text=[text],
        images=[image],
        return_tensors="pt"
    ).to(model.device)

    output = model.generate(
        **inputs,
        max_new_tokens=60
    )

    answer = processor.batch_decode(
    output,
    skip_special_tokens=True
)[0]

# Keep only the assistant's reply
if "assistant" in answer:
    answer = answer.split("assistant")[-1].strip()

print("="*60)
print(f"Frame: {frame}")
print(f"Question: {question}")
print(f"Answer: {answer}")

Ask a question about the video: What food is being prepared?
Frame: frame_9.jpg
Question: What food is being prepared?
Answer: The image shows a cooking process where a sauce or curry is being prepared in a pan. The ingredients visible include spices and possibly some vegetables or meat. The specific dish cannot be determined from the image alone, but it appears to be a type of curry or stew.
